#### Steps taken:
1. Read bronze sprints data.
2. Keep only columns required for analytics, will drop the url column.
3. Standardize the column name using snake_case (ex: driverID -> driver_id, positionText -> finish_position_text etc.)
4. Rename columns to make it more meaningful.
5. Filter out rows where season, round, constructor_id, driver_id that is null (business key validation)
5. Remove duplicate records.
6. Transform values of columns to title case (ex: race_name).
7. Write the transform data to silver results table.

In [0]:
%run ../environment_config

In [0]:
bronze_table = f"{catalog}.{bronze_schema}.sprints"
silver_table = f"{catalog}.{silver_schema}.sprints"

In [0]:
from pyspark.sql import functions as F

In [0]:
sprints_df = spark.read.table(bronze_table)\
                  .select("*").drop("url")\
                  .withColumnsRenamed({
                         "constructorId" : "constructor_id",
                         "date" : "race_date",
                         "driverID" : "driver_id",
                         "raceName" : "race_name",
                         "grid" : "grid_position",
                         "laps" : "completed_laps",
                         "number" : "car_number",
                         "position" : "final_position",
                         "positionText" : "final_position_text"
                  })

In [0]:
sprints_valid_df = sprints_df.filter(
                    F.col("season").isNotNull() &
                    F.col("round").isNotNull() &
                    F.col("constructor_id").isNotNull() &
                    F.col("driver_id").isNotNull() 
                     )

In [0]:
display(sprints_df.count() - sprints_valid_df.count())

7

In [0]:
display(sprints_valid_df.count())

493

In [0]:
sprints_distinct_df = sprints_valid_df.dropDuplicates(
                         ["season", "round", "driver_id", "constructor_id"]
                     )

In [0]:
display(sprints_distinct_df.count())

480

In [0]:
display(sprints_valid_df.count() - sprints_distinct_df.count())

13

In [0]:
sprints_final_df = sprints_distinct_df.withColumn(
                                "race_name", F.initcap(F.col("race_name"))
)

In [0]:
(
    sprints_final_df.write
                    .format("delta")
                    .mode("overwrite")
                    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))

constructor_id,race_date,driver_id,grid_position,completed_laps,car_number,points,final_position,final_position_text,race_name,round,season,status,ingestion_timestamp,souce_file,source_path
mercedes,2021-07-18,hamilton,1,17,44,2.0,2,2,British Grand Prix,10,2021,Finished,2026-07-20T16:32:52.022Z,sprints_2021.json,dbfs:/Volumes/formula1/landing/lvolumne/sprints/sprints_2021.json
red_bull,2021-11-14,perez,3,24,11,0.0,4,4,São Paulo Grand Prix,19,2021,Finished,2026-07-20T16:32:52.022Z,sprints_2021.json,dbfs:/Volumes/formula1/landing/lvolumne/sprints/sprints_2021.json
ferrari,2021-11-14,leclerc,6,24,16,0.0,7,7,São Paulo Grand Prix,19,2021,Finished,2026-07-20T16:32:52.022Z,sprints_2021.json,dbfs:/Volumes/formula1/landing/lvolumne/sprints/sprints_2021.json
mercedes,2023-04-30,hamilton,6,17,44,2.0,7,7,Azerbaijan Grand Prix,4,2023,Finished,2026-07-20T16:32:52.022Z,sprints_2023.json,dbfs:/Volumes/formula1/landing/lvolumne/sprints/sprints_2023.json
mclaren,2023-07-02,norris,3,24,4,0.0,9,9,Austrian Grand Prix,9,2023,Finished,2026-07-20T16:32:52.022Z,sprints_2023.json,dbfs:/Volumes/formula1/landing/lvolumne/sprints/sprints_2023.json
mercedes,2023-07-02,hamilton,18,24,44,0.0,10,10,Austrian Grand Prix,9,2023,Finished,2026-07-20T16:32:52.022Z,sprints_2023.json,dbfs:/Volumes/formula1/landing/lvolumne/sprints/sprints_2023.json
alpine,2023-07-30,gasly,6,11,10,6.0,3,3,Belgian Grand Prix,12,2023,Finished,2026-07-20T16:32:52.022Z,sprints_2023.json,dbfs:/Volumes/formula1/landing/lvolumne/sprints/sprints_2023.json
mclaren,2023-07-30,norris,5,11,4,3.0,6,6,Belgian Grand Prix,12,2023,Finished,2026-07-20T16:32:52.022Z,sprints_2023.json,dbfs:/Volumes/formula1/landing/lvolumne/sprints/sprints_2023.json
alphatauri,2023-10-22,ricciardo,10,19,3,0.0,12,12,United States Grand Prix,18,2023,Finished,2026-07-20T16:32:52.022Z,sprints_2023.json,dbfs:/Volumes/formula1/landing/lvolumne/sprints/sprints_2023.json
aston_martin,2023-10-22,stroll,14,16,18,0.0,20,R,United States Grand Prix,18,2023,Brakes,2026-07-20T16:32:52.022Z,sprints_2023.json,dbfs:/Volumes/formula1/landing/lvolumne/sprints/sprints_2023.json
